# Step 2a v2 — Private fork + reproducible Kaggle RegDB setup

This notebook replaces the previous Step 2 / Step 2b notebooks for now.

Scope of this notebook:

- Clone/pull your **private fork** using a Kaggle Secret named `GITHUB_TOKEN`.
- Switch to `feature/upr-cre`.
- Create reproducible Kaggle helper scripts.
- Prepare RegDB from Kaggle input into the layout expected by WSL-VIReID.
- Apply only minimal Kaggle compatibility fixes.
- Optionally run a short RegDB smoke test.
- Commit and push Step 2a files back to your private fork.

This notebook does **not** implement UPR-CRE, relation diagnostics, AMP/FP16, DDP, or T4x2 parallel execution. Those will be separate steps after this setup is stable.


In [1]:

# =============================
# Step 2a central configuration
# =============================
from pathlib import Path
import os

CFG = {
    # Your private fork
    "GITHUB_USERNAME": "TranTruongMMCII",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",

    # Kaggle paths
    "WORK_DIR": "/kaggle/working",
    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",

    # Leave empty for auto-detection under /kaggle/input.
    # Your current dataset should auto-detect at:
    # /kaggle/input/datasets/xqq027/reg-db/RegDB
    "REGDB_SOURCE": "/kaggle/input/datasets/xqq027/reg-db/RegDB",

    # Kaggle Secret name that stores your GitHub Personal Access Token.
    "GITHUB_TOKEN_SECRET": "GITHUB_TOKEN",

    # Git commit identity. Change email before committing.
    "GIT_USER_NAME": "TranTruongMMCII",
    "GIT_USER_EMAIL": "truongtlt147@gmail.com",

    # Smoke test is optional because it takes time.
    "RUN_SMOKE_TEST": False,
}

WORK_DIR = Path(CFG["WORK_DIR"])
REPO_DIR = WORK_DIR / CFG["REPO_NAME"]
WSL_DIR = REPO_DIR / "WSL_ReID"
PUBLIC_REPO_URL = f"https://github.com/{CFG['GITHUB_USERNAME']}/{CFG['REPO_NAME']}.git"

print("Public repo URL:", PUBLIC_REPO_URL)
print("Repo dir:", REPO_DIR)
print("WSL dir:", WSL_DIR)
print("Branch:", CFG["BRANCH"])
print("Data root:", CFG["DATA_ROOT"])


Public repo URL: https://github.com/TranTruongMMCII/WSL-VIReID.git
Repo dir: /kaggle/working/WSL-VIReID
WSL dir: /kaggle/working/WSL-VIReID/WSL_ReID
Branch: feature/upr-cre
Data root: /kaggle/working/VIREID_Dataset


## 1. Private GitHub helper

This cell uses `GIT_ASKPASS` so the token is **not embedded in the remote URL** and is not printed by `git remote -v`.

Before running, create a Kaggle Secret:

```text
Name:  GITHUB_TOKEN
Value: <your GitHub Personal Access Token>
```

The token needs access to your private fork.


In [2]:

import subprocess
from pathlib import Path


def run(cmd, cwd=None, env=None, check=True, capture=False):
    """Run a command without printing secrets."""
    cwd = str(cwd) if cwd is not None else None
    printable = " ".join(str(x) for x in cmd)
    print(f"$ {printable}" + (f"  # cwd={cwd}" if cwd else ""))
    if capture:
        return subprocess.run(cmd, cwd=cwd, env=env, check=check, text=True, capture_output=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)


def get_github_token(secret_name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception as exc:
        raise RuntimeError("This notebook expects to run on Kaggle with kaggle_secrets available.") from exc
    token = UserSecretsClient().get_secret(secret_name)
    if not token or len(token.strip()) < 10:
        raise RuntimeError(f"Kaggle Secret {secret_name!r} is missing or too short.")
    return token.strip()


def make_git_env(token: str):
    """Create a Git environment that authenticates via GIT_ASKPASS."""
    askpass = Path(CFG["WORK_DIR"]) / ".git_askpass_github.sh"
    askpass.write_text("""#!/bin/sh
case "$1" in
  *Username*) printf "%s" "$GITHUB_USERNAME" ;;
  *Password*) printf "%s" "$GITHUB_TOKEN" ;;
  *) printf "" ;;
esac
""")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["GITHUB_USERNAME"] = CFG["GITHUB_USERNAME"]
    env["GITHUB_TOKEN"] = token
    return env

GITHUB_TOKEN = get_github_token(CFG["GITHUB_TOKEN_SECRET"])
GIT_ENV = make_git_env(GITHUB_TOKEN)
print("GitHub token loaded from Kaggle Secret. Token will not be printed.")


GitHub token loaded from Kaggle Secret. Token will not be printed.


## 2. Clone or update your private fork

This cell is branch-safe:

- If the repo does not exist in `/kaggle/working`, it clones it.
- If the repo already exists, it fetches from origin.
- If `feature/upr-cre` exists on GitHub, it tracks that branch.
- If it does not exist yet, it creates it locally.


In [3]:

WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

if not REPO_DIR.exists():
    run(["git", "clone", PUBLIC_REPO_URL, str(REPO_DIR)], env=GIT_ENV)
else:
    print("Repo already exists:", REPO_DIR)

# Always keep remote URL public, not tokenized.
run(["git", "remote", "set-url", "origin", PUBLIC_REPO_URL], cwd=REPO_DIR)

# Fetch using token through GIT_ASKPASS.
run(["git", "fetch", "origin"], cwd=REPO_DIR, env=GIT_ENV)

branch = CFG["BRANCH"]
res = run(["git", "ls-remote", "--heads", "origin", branch], cwd=REPO_DIR, env=GIT_ENV, capture=True)
remote_branch_exists = bool(res.stdout.strip())
print("Remote branch exists:", remote_branch_exists)

# Switch branch robustly.
local_branches = run(["git", "branch", "--list", branch], cwd=REPO_DIR, capture=True).stdout.strip()
if local_branches:
    run(["git", "switch", branch], cwd=REPO_DIR)
elif remote_branch_exists:
    run(["git", "switch", "-c", branch, "--track", f"origin/{branch}"], cwd=REPO_DIR)
else:
    run(["git", "switch", "-c", branch], cwd=REPO_DIR)

# Pull only if remote branch exists.
if remote_branch_exists:
    run(["git", "pull", "--ff-only", "origin", branch], cwd=REPO_DIR, env=GIT_ENV, check=False)

# Configure commit identity locally for this repo only.
run(["git", "config", "user.name", CFG["GIT_USER_NAME"]], cwd=REPO_DIR)
run(["git", "config", "user.email", CFG["GIT_USER_EMAIL"]], cwd=REPO_DIR)

run(["git", "branch", "--show-current"], cwd=REPO_DIR)
run(["git", "remote", "-v"], cwd=REPO_DIR)
run(["git", "status", "--short"], cwd=REPO_DIR)


$ git clone https://github.com/TranTruongMMCII/WSL-VIReID.git /kaggle/working/WSL-VIReID


Cloning into '/kaggle/working/WSL-VIReID'...


$ git remote set-url origin https://github.com/TranTruongMMCII/WSL-VIReID.git  # cwd=/kaggle/working/WSL-VIReID
$ git fetch origin  # cwd=/kaggle/working/WSL-VIReID
$ git ls-remote --heads origin feature/upr-cre  # cwd=/kaggle/working/WSL-VIReID
Remote branch exists: True
$ git branch --list feature/upr-cre  # cwd=/kaggle/working/WSL-VIReID
$ git switch -c feature/upr-cre --track origin/feature/upr-cre  # cwd=/kaggle/working/WSL-VIReID
Branch 'feature/upr-cre' set up to track remote branch 'feature/upr-cre' from 'origin'.
$ git pull --ff-only origin feature/upr-cre  # cwd=/kaggle/working/WSL-VIReID


Switched to a new branch 'feature/upr-cre'


Already up to date.
$ git config user.name TranTruongMMCII  # cwd=/kaggle/working/WSL-VIReID
$ git config user.email truongtlt147@gmail.com  # cwd=/kaggle/working/WSL-VIReID
$ git branch --show-current  # cwd=/kaggle/working/WSL-VIReID
feature/upr-cre
$ git remote -v  # cwd=/kaggle/working/WSL-VIReID
origin	https://github.com/TranTruongMMCII/WSL-VIReID.git (fetch)
origin	https://github.com/TranTruongMMCII/WSL-VIReID.git (push)
$ git status --short  # cwd=/kaggle/working/WSL-VIReID


From https://github.com/TranTruongMMCII/WSL-VIReID
 * branch            feature/upr-cre -> FETCH_HEAD


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

## 3. Verify Step 0/1 docs exist

Step 2a assumes the documentation from Step 0/1 is already on `feature/upr-cre`.


In [4]:

required_docs = [
    "docs/research_positioning.md",
    "docs/original_wsl_settings.md",
    "docs/experiments/regdb_dev_baseline.md",
    "docs/working_protocol.md",
]

missing = []
for rel in required_docs:
    p = REPO_DIR / rel
    print(f"{rel:<45}", "OK" if p.exists() else "MISSING")
    if not p.exists():
        missing.append(rel)

assert not missing, "Missing Step 0/1 docs: " + ", ".join(missing)


docs/research_positioning.md                  OK
docs/original_wsl_settings.md                 OK
docs/experiments/regdb_dev_baseline.md        OK
docs/working_protocol.md                      OK


## 4. Create Step 2a files

Naming convention used in this notebook:

```text
CFG["WORK_DIR"]     -> Kaggle working directory
REPO_DIR            -> /kaggle/working/WSL-VIReID
WSL_DIR             -> /kaggle/working/WSL-VIReID/WSL_ReID
CFG["DATA_ROOT"]    -> /kaggle/working/VIREID_Dataset
CFG["REGDB_SOURCE"] -> source RegDB folder under /kaggle/input, or empty for auto-detect
```


In [5]:

from pathlib import Path

FILE_MANIFEST = {
  "WSL_ReID/requirements-kaggle.txt": "# Minimal Kaggle requirements for WSL-VIReID development.\n# Do not install the original requirements.txt on Kaggle because it pins Python/Torch.\nftfy\nregex\nsetproctitle\nscikit-learn\ntqdm\n",
  "WSL_ReID/scripts/prepare_regdb_kaggle.py": "#!/usr/bin/env python3\n\"\"\"Prepare RegDB Kaggle dataset layout for WSL-VIReID.\n\nThe original RegDB loader expects:\n\n    <data-root>/RegDB/idx/train_visible_1.txt\n    <data-root>/RegDB/idx/train_thermal_1.txt\n    <data-root>/RegDB/Visible/...\n    <data-root>/RegDB/Thermal/...\n\nThis script creates that layout under /kaggle/working without copying image files.\nIt uses symlinks to the read-only Kaggle input folder.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport shutil\nfrom pathlib import Path\n\n\ndef find_regdb_source(input_root: Path = Path(\"/kaggle/input\")) -> Path:\n    candidates = []\n    for p in input_root.rglob(\"RegDB\"):\n        if not p.is_dir():\n            continue\n        idx_ok = (p / \"idx\").exists()\n        visible_ok = (p / \"Visible\").exists() or (p / \"visible\").exists()\n        thermal_ok = (p / \"Thermal\").exists() or (p / \"thermal\").exists()\n        if idx_ok and visible_ok and thermal_ok:\n            candidates.append(p)\n    if not candidates:\n        raise FileNotFoundError(\"Could not find RegDB under /kaggle/input. Attach the RegDB Kaggle dataset first.\")\n    candidates = sorted(candidates, key=lambda x: len(str(x)))\n    return candidates[0]\n\n\ndef remove_path(p: Path) -> None:\n    if p.is_symlink() or p.is_file():\n        p.unlink()\n    elif p.exists():\n        shutil.rmtree(p)\n\n\ndef symlink_or_copytree(src: Path, dst: Path) -> None:\n    remove_path(dst)\n    try:\n        dst.symlink_to(src, target_is_directory=True)\n    except OSError:\n        shutil.copytree(src, dst)\n\n\ndef normalize_index_line(raw: str, expected_modality: str) -> str | None:\n    raw = raw.strip()\n    if not raw:\n        return None\n    parts = raw.split()\n    if len(parts) < 2:\n        raise ValueError(f\"Invalid index line: {raw!r}\")\n\n    rel = parts[0].replace(\"\\\\\", \"/\")\n    label = parts[1]\n\n    while rel.startswith(\"./\"):\n        rel = rel[2:]\n    if rel.startswith(\"RegDB/\"):\n        rel = rel[len(\"RegDB/\"):]\n\n    chunks = [x for x in rel.split(\"/\") if x]\n    if not chunks:\n        raise ValueError(f\"Invalid relative image path: {rel!r}\")\n\n    first = chunks[0].lower()\n    if first == \"visible\":\n        chunks[0] = \"Visible\"\n    elif first in {\"thermal\", \"infrared\", \"ir\"}:\n        chunks[0] = \"Thermal\"\n    else:\n        chunks.insert(0, expected_modality)\n\n    return \"/\".join(chunks) + \" \" + label\n\n\ndef prepare_regdb(regdb_source: Path, data_root: Path) -> Path:\n    regdb_source = regdb_source.resolve()\n    if not regdb_source.exists():\n        raise FileNotFoundError(f\"RegDB source does not exist: {regdb_source}\")\n\n    visible_src = regdb_source / \"Visible\" if (regdb_source / \"Visible\").exists() else regdb_source / \"visible\"\n    thermal_src = regdb_source / \"Thermal\" if (regdb_source / \"Thermal\").exists() else regdb_source / \"thermal\"\n    idx_src = regdb_source / \"idx\"\n\n    if not visible_src.exists() or not thermal_src.exists() or not idx_src.exists():\n        raise FileNotFoundError(f\"RegDB source is missing Visible/Thermal/idx folders: {regdb_source}\")\n\n    if data_root.is_symlink() or data_root.is_file():\n        data_root.unlink()\n    data_root.mkdir(parents=True, exist_ok=True)\n\n    regdb_dst = data_root / \"RegDB\"\n    regdb_dst.mkdir(parents=True, exist_ok=True)\n\n    symlink_or_copytree(visible_src, regdb_dst / \"Visible\")\n    symlink_or_copytree(thermal_src, regdb_dst / \"Thermal\")\n    symlink_or_copytree(visible_src, regdb_dst / \"visible\")\n    symlink_or_copytree(thermal_src, regdb_dst / \"thermal\")\n\n    idx_dst = regdb_dst / \"idx\"\n    remove_path(idx_dst)\n    idx_dst.mkdir(parents=True, exist_ok=True)\n\n    for txt in sorted(idx_src.glob(\"*.txt\")):\n        expected = \"Thermal\" if \"thermal\" in txt.name.lower() else \"Visible\"\n        lines = []\n        for raw in txt.read_text().splitlines():\n            line = normalize_index_line(raw, expected)\n            if line is not None:\n                lines.append(line)\n        (idx_dst / txt.name).write_text(\"\\n\".join(lines) + \"\\n\")\n\n    required = [\n        idx_dst / \"train_visible_1.txt\",\n        idx_dst / \"train_thermal_1.txt\",\n        idx_dst / \"test_visible_1.txt\",\n        idx_dst / \"test_thermal_1.txt\",\n    ]\n    missing = [str(p) for p in required if not p.exists()]\n    if missing:\n        raise FileNotFoundError(\"Missing required RegDB index files: \" + \", \".join(missing))\n\n    for f in required:\n        first = f.read_text().splitlines()[0]\n        rel = first.split()[0]\n        img_path = regdb_dst / rel\n        if not img_path.exists():\n            raise FileNotFoundError(f\"Index file {f.name} points to missing image: {img_path}\")\n\n    return regdb_dst\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--regdb-source\", default=\"\", help=\"Path to source RegDB folder. Empty means auto-detect under /kaggle/input.\")\n    parser.add_argument(\"--data-root\", default=\"/kaggle/working/VIREID_Dataset\", help=\"Output data root expected by --data-path.\")\n    args = parser.parse_args()\n\n    regdb_source = Path(args.regdb_source) if args.regdb_source else find_regdb_source()\n    data_root = Path(args.data_root)\n    regdb_dst = prepare_regdb(regdb_source, data_root)\n\n    print(\"RegDB source:\", regdb_source)\n    print(\"RegDB prepared at:\", regdb_dst)\n    print(\"Use training argument: --data-path\", data_root)\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "WSL_ReID/scripts/check_kaggle_env.py": "#!/usr/bin/env python3\n\"\"\"Check Kaggle GPU and RegDB layout before training.\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\nfrom PIL import Image\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--data-root\", default=\"/kaggle/working/VIREID_Dataset\")\n    args = parser.parse_args()\n\n    print(\"=== PyTorch / GPU ===\")\n    try:\n        import torch\n        print(\"torch:\", torch.__version__)\n        print(\"cuda available:\", torch.cuda.is_available())\n        print(\"gpu count:\", torch.cuda.device_count())\n        for i in range(torch.cuda.device_count()):\n            print(f\"gpu {i}:\", torch.cuda.get_device_name(i))\n    except Exception as exc:\n        print(\"torch check failed:\", repr(exc))\n\n    print(\"\\n=== RegDB layout ===\")\n    base = Path(args.data_root) / \"RegDB\"\n    required = [\n        base / \"idx/train_visible_1.txt\",\n        base / \"idx/train_thermal_1.txt\",\n        base / \"idx/test_visible_1.txt\",\n        base / \"idx/test_thermal_1.txt\",\n    ]\n    for f in required:\n        print(f, \"OK\" if f.exists() else \"MISSING\")\n        if not f.exists():\n            raise FileNotFoundError(f)\n\n    for f in required:\n        first = f.read_text().splitlines()[0]\n        rel, label = first.split()[:2]\n        img_path = base / rel\n        print(\"\\n\", f.name)\n        print(\"first line:\", first)\n        print(\"image:\", img_path)\n        print(\"exists:\", img_path.exists())\n        if not img_path.exists():\n            raise FileNotFoundError(img_path)\n        img = Image.open(img_path)\n        print(\"image size:\", img.size, \"mode:\", img.mode, \"label:\", label)\n\n    print(\"\\nEnvironment and RegDB layout OK.\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "WSL_ReID/scripts/apply_kaggle_compat_patches.py": "#!/usr/bin/env python3\n\"\"\"Apply minimal Kaggle compatibility patches.\n\nThese patches do not implement UPR-CRE. They only fix runtime issues observed on Kaggle:\n\n1. Weak_loss(method='log') mismatch with current Weak_loss signature.\n2. Pillow Image.ANTIALIAS removal in newer Pillow versions.\n\"\"\"\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\n\ndef patch_file(path: Path, replacements: list[tuple[str, str]]) -> bool:\n    if not path.exists():\n        return False\n    text = path.read_text()\n    new_text = text\n    for old, new in replacements:\n        new_text = new_text.replace(old, new)\n    if new_text != text:\n        path.write_text(new_text)\n        return True\n    return False\n\n\ndef main() -> None:\n    wsl_dir = Path(__file__).resolve().parents[1]\n    changed = []\n\n    p = wsl_dir / \"models\" / \"__init__.py\"\n    if patch_file(p, [(\"Weak_loss(method='log')\", \"Weak_loss()\")]):\n        changed.append(str(p.relative_to(wsl_dir)))\n\n    antialias_replacement = \"(Image.Resampling.LANCZOS if hasattr(Image, 'Resampling') else Image.LANCZOS)\"\n    for py in wsl_dir.rglob(\"*.py\"):\n        if \"scripts/apply_kaggle_compat_patches.py\" in str(py):\n            continue\n        if patch_file(py, [(\"Image.ANTIALIAS\", antialias_replacement)]):\n            changed.append(str(py.relative_to(wsl_dir)))\n\n    if changed:\n        print(\"Patched files:\")\n        for x in changed:\n            print(\" -\", x)\n    else:\n        print(\"No compatibility patches were needed.\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "WSL_ReID/scripts/run_regdb_smoke.sh": "#!/usr/bin/env bash\nset -euo pipefail\n\nTHIS_DIR=\"$(cd \"$(dirname \"${BASH_SOURCE[0]}\")\" && pwd)\"\nWSL_DIR=\"$(cd \"${THIS_DIR}/..\" && pwd)\"\ncd \"${WSL_DIR}\"\n\nDATA_ROOT=\"${DATA_ROOT:-/kaggle/working/VIREID_Dataset}\"\nREGDB_SOURCE=\"${REGDB_SOURCE:-}\"\nRUN_NAME=\"${RUN_NAME:-smoke_regdb_step2a}\"\nDEVICE=\"${DEVICE:-0}\"\nSEED=\"${SEED:-1}\"\n\npython -m pip install -q -r requirements-kaggle.txt\npython scripts/apply_kaggle_compat_patches.py\n\nPREPARE_ARGS=(--data-root \"${DATA_ROOT}\")\nif [[ -n \"${REGDB_SOURCE}\" ]]; then\n  PREPARE_ARGS+=(--regdb-source \"${REGDB_SOURCE}\")\nfi\npython scripts/prepare_regdb_kaggle.py \"${PREPARE_ARGS[@]}\"\npython scripts/check_kaggle_env.py --data-root \"${DATA_ROOT}\"\n\npython main.py \\\n  --dataset regdb \\\n  --data-path \"${DATA_ROOT}\" \\\n  --debug wsl \\\n  --save-path \"${RUN_NAME}\" \\\n  --arch resnet \\\n  --trial 1 \\\n  --stage1-epoch 1 \\\n  --stage2-epoch 2 \\\n  --milestones 1 2 \\\n  --lr 0.00045 \\\n  --batch-pidnum 2 \\\n  --pid-numsample 2 \\\n  --test-batch 32 \\\n  --num-workers 2 \\\n  --device \"${DEVICE}\" \\\n  --seed \"${SEED}\"\n",
  "WSL_ReID/scripts/run_regdb_dev_baseline.sh": "#!/usr/bin/env bash\nset -euo pipefail\n\nTHIS_DIR=\"$(cd \"$(dirname \"${BASH_SOURCE[0]}\")\" && pwd)\"\nWSL_DIR=\"$(cd \"${THIS_DIR}/..\" && pwd)\"\ncd \"${WSL_DIR}\"\n\nDATA_ROOT=\"${DATA_ROOT:-/kaggle/working/VIREID_Dataset}\"\nREGDB_SOURCE=\"${REGDB_SOURCE:-}\"\nRUN_NAME=\"${RUN_NAME:-baseline_regdb_s5_s15_step2a}\"\nDEVICE=\"${DEVICE:-0}\"\nSEED=\"${SEED:-1}\"\nSTAGE1_EPOCH=\"${STAGE1_EPOCH:-5}\"\nSTAGE2_EPOCH=\"${STAGE2_EPOCH:-15}\"\nBATCH_PIDNUM=\"${BATCH_PIDNUM:-5}\"\nPID_NUMSAMPLE=\"${PID_NUMSAMPLE:-4}\"\nTEST_BATCH=\"${TEST_BATCH:-64}\"\nNUM_WORKERS=\"${NUM_WORKERS:-2}\"\n\npython -m pip install -q -r requirements-kaggle.txt\npython scripts/apply_kaggle_compat_patches.py\n\nPREPARE_ARGS=(--data-root \"${DATA_ROOT}\")\nif [[ -n \"${REGDB_SOURCE}\" ]]; then\n  PREPARE_ARGS+=(--regdb-source \"${REGDB_SOURCE}\")\nfi\npython scripts/prepare_regdb_kaggle.py \"${PREPARE_ARGS[@]}\"\npython scripts/check_kaggle_env.py --data-root \"${DATA_ROOT}\"\n\npython main.py \\\n  --dataset regdb \\\n  --data-path \"${DATA_ROOT}\" \\\n  --debug wsl \\\n  --save-path \"${RUN_NAME}\" \\\n  --arch resnet \\\n  --trial 1 \\\n  --stage1-epoch \"${STAGE1_EPOCH}\" \\\n  --stage2-epoch \"${STAGE2_EPOCH}\" \\\n  --milestones 8 12 \\\n  --lr 0.00045 \\\n  --batch-pidnum \"${BATCH_PIDNUM}\" \\\n  --pid-numsample \"${PID_NUMSAMPLE}\" \\\n  --test-batch \"${TEST_BATCH}\" \\\n  --num-workers \"${NUM_WORKERS}\" \\\n  --device \"${DEVICE}\" \\\n  --seed \"${SEED}\"\n",
  "docs/experiments/step2a_kaggle_private_env.md": "# Step 2a — Private GitHub + reproducible Kaggle RegDB setup\n\n## Purpose\n\nThis step makes the Kaggle development environment reproducible for the private fork.\nIt does not implement UPR-CRE and does not modify the training objective.\n\n## Standard variables\n\n| Variable | Meaning |\n|---|---|\n| `WORK_DIR` | `/kaggle/working` |\n| `REPO_DIR` | `/kaggle/working/WSL-VIReID` |\n| `WSL_DIR` | `/kaggle/working/WSL-VIReID/WSL_ReID` |\n| `DATA_ROOT` | `/kaggle/working/VIREID_Dataset` |\n| `REGDB_SOURCE` | source RegDB folder under `/kaggle/input`, or empty for auto-detect |\n\n## Files added\n\n```text\nWSL_ReID/requirements-kaggle.txt\nWSL_ReID/scripts/prepare_regdb_kaggle.py\nWSL_ReID/scripts/check_kaggle_env.py\nWSL_ReID/scripts/apply_kaggle_compat_patches.py\nWSL_ReID/scripts/run_regdb_smoke.sh\nWSL_ReID/scripts/run_regdb_dev_baseline.sh\ndocs/experiments/step2a_kaggle_private_env.md\n```\n\n## Minimal runtime patches\n\nThe compatibility patch script only fixes Kaggle/runtime issues:\n\n1. `Weak_loss(method='log')` -> `Weak_loss()` because the current `Weak_loss` class does not accept `method`.\n2. `Image.ANTIALIAS` -> a Pillow-compatible LANCZOS expression.\n\nThese are not UPR-CRE method changes.\n\n## Smoke test command\n\n```bash\ncd /kaggle/working/WSL-VIReID/WSL_ReID\nDATA_ROOT=/kaggle/working/VIREID_Dataset REGDB_SOURCE=/kaggle/input/datasets/xqq027/reg-db/RegDB bash scripts/run_regdb_smoke.sh\n```\n\nIf `REGDB_SOURCE` is empty, the prepare script auto-detects RegDB under `/kaggle/input`.\n\n## Development baseline command\n\n```bash\ncd /kaggle/working/WSL-VIReID/WSL_ReID\nDATA_ROOT=/kaggle/working/VIREID_Dataset RUN_NAME=baseline_regdb_s5_s15_step2a bash scripts/run_regdb_dev_baseline.sh\n```\n\nThe development baseline can take several hours on T4. It is not required every time if a previous local baseline already exists.\n"
}

for rel, content in FILE_MANIFEST.items():
    path = REPO_DIR / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
    if path.suffix in {".sh", ".py"}:
        path.chmod(0o755)
    print("wrote", rel)

print("Step 2a files created.")


wrote WSL_ReID/requirements-kaggle.txt
wrote WSL_ReID/scripts/prepare_regdb_kaggle.py
wrote WSL_ReID/scripts/check_kaggle_env.py
wrote WSL_ReID/scripts/apply_kaggle_compat_patches.py
wrote WSL_ReID/scripts/run_regdb_smoke.sh
wrote WSL_ReID/scripts/run_regdb_dev_baseline.sh
wrote docs/experiments/step2a_kaggle_private_env.md
Step 2a files created.


## 5. Apply compatibility patches, prepare RegDB, and check environment

This cell does not train. It verifies that the scripts work on the current Kaggle session.


In [6]:

# Install minimal packages. This does not downgrade Kaggle PyTorch.
run(["python", "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=WSL_DIR)

# Apply compatibility patches. These may modify models/__init__.py and dataset files.
run(["python", "scripts/apply_kaggle_compat_patches.py"], cwd=WSL_DIR)

# Prepare RegDB. If REGDB_SOURCE is empty, the script auto-detects under /kaggle/input.
prepare_cmd = ["python", "scripts/prepare_regdb_kaggle.py", "--data-root", CFG["DATA_ROOT"]]
if CFG["REGDB_SOURCE"]:
    prepare_cmd += ["--regdb-source", CFG["REGDB_SOURCE"]]
run(prepare_cmd, cwd=WSL_DIR)

# Check GPU and dataset layout.
run(["python", "scripts/check_kaggle_env.py", "--data-root", CFG["DATA_ROOT"]], cwd=WSL_DIR)


$ python -m pip install -q -r requirements-kaggle.txt  # cwd=/kaggle/working/WSL-VIReID/WSL_ReID
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
$ python scripts/apply_kaggle_compat_patches.py  # cwd=/kaggle/working/WSL-VIReID/WSL_ReID
Patched files:
 - models/__init__.py
 - datasets/llcm.py
 - datasets/sysu.py
 - datasets/regdb.py
$ python scripts/prepare_regdb_kaggle.py --data-root /kaggle/working/VIREID_Dataset --regdb-source /kaggle/input/datasets/xqq027/reg-db/RegDB  # cwd=/kaggle/working/WSL-VIReID/WSL_ReID
RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
$ python scripts/check_kaggle_env.py --data-root /kaggle/working/VIREID_Dataset  # cwd=/kaggle/working/WSL-VIReID/WSL_ReID
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIR

CompletedProcess(args=['python', 'scripts/check_kaggle_env.py', '--data-root', '/kaggle/working/VIREID_Dataset'], returncode=0)

## 6. Optional smoke test

This takes time. Keep `RUN_SMOKE_TEST=False` if you only want to commit the setup scripts first.


In [7]:

if CFG["RUN_SMOKE_TEST"]:
    env = os.environ.copy()
    env["DATA_ROOT"] = CFG["DATA_ROOT"]
    if CFG["REGDB_SOURCE"]:
        env["REGDB_SOURCE"] = CFG["REGDB_SOURCE"]
    env["RUN_NAME"] = "smoke_regdb_step2a"
    env["DEVICE"] = "0"
    run(["bash", "scripts/run_regdb_smoke.sh"], cwd=WSL_DIR, env=env)
else:
    print("Skipping smoke test. Set CFG['RUN_SMOKE_TEST'] = True to run it.")


Skipping smoke test. Set CFG['RUN_SMOKE_TEST'] = True to run it.


## 7. Inspect changes before commit


In [8]:

run(["git", "status", "--short"], cwd=REPO_DIR)
print("\nDiff summary:")
run(["git", "diff", "--stat"], cwd=REPO_DIR, check=False)


$ git status --short  # cwd=/kaggle/working/WSL-VIReID
 M WSL_ReID/datasets/llcm.py
 M WSL_ReID/datasets/regdb.py
 M WSL_ReID/datasets/sysu.py
 M WSL_ReID/models/__init__.py
?? WSL_ReID/requirements-kaggle.txt
?? WSL_ReID/scripts/
?? docs/experiments/step2a_kaggle_private_env.md

Diff summary:
$ git diff --stat  # cwd=/kaggle/working/WSL-VIReID
 WSL_ReID/datasets/llcm.py   | 4 ++--
 WSL_ReID/datasets/regdb.py  | 4 ++--
 WSL_ReID/datasets/sysu.py   | 2 +-
 WSL_ReID/models/__init__.py | 2 +-
 4 files changed, 6 insertions(+), 6 deletions(-)


CompletedProcess(args=['git', 'diff', '--stat'], returncode=0)

## 8. Commit and push Step 2a

Before running this cell, set a real email in `CFG["GIT_USER_EMAIL"]`.

This cell pushes using the Kaggle Secret token through `GIT_ASKPASS`. The Git remote stays public and does not store the token.


In [10]:

if CFG["GIT_USER_EMAIL"] == "your-email@example.com":
    raise RuntimeError("Please set CFG['GIT_USER_EMAIL'] to your real commit email before committing.")

run(["git", "config", "user.name", CFG["GIT_USER_NAME"]], cwd=REPO_DIR)
run(["git", "config", "user.email", CFG["GIT_USER_EMAIL"]], cwd=REPO_DIR)

files_to_add = [
    "WSL_ReID/requirements-kaggle.txt",
    "WSL_ReID/scripts/prepare_regdb_kaggle.py",
    "WSL_ReID/scripts/check_kaggle_env.py",
    "WSL_ReID/scripts/apply_kaggle_compat_patches.py",
    "WSL_ReID/scripts/run_regdb_smoke.sh",
    "WSL_ReID/scripts/run_regdb_dev_baseline.sh",
    "docs/experiments/step2a_kaggle_private_env.md",
    # Compatibility patches may modify these files.
    "WSL_ReID/models/__init__.py",
    "WSL_ReID/datasets/regdb.py",
    "WSL_ReID/datasets/sysu.py",
    "WSL_ReID/datasets/llcm.py",
]

run(["git", "add", "-f"] + files_to_add, cwd=REPO_DIR)
status = run(["git", "status", "--short"], cwd=REPO_DIR, capture=True).stdout.strip()
print(status)

if status:
    run(["git", "commit", "-m", "chore: add private Kaggle RegDB setup scripts"], cwd=REPO_DIR)
else:
    print("No changes to commit.")

run(["git", "push", "-u", "origin", CFG["BRANCH"]], cwd=REPO_DIR, env=GIT_ENV)
run(["git", "remote", "-v"], cwd=REPO_DIR)


$ git config user.name TranTruongMMCII  # cwd=/kaggle/working/WSL-VIReID
$ git config user.email truongtlt147@gmail.com  # cwd=/kaggle/working/WSL-VIReID
$ git add -f WSL_ReID/requirements-kaggle.txt WSL_ReID/scripts/prepare_regdb_kaggle.py WSL_ReID/scripts/check_kaggle_env.py WSL_ReID/scripts/apply_kaggle_compat_patches.py WSL_ReID/scripts/run_regdb_smoke.sh WSL_ReID/scripts/run_regdb_dev_baseline.sh docs/experiments/step2a_kaggle_private_env.md WSL_ReID/models/__init__.py WSL_ReID/datasets/regdb.py WSL_ReID/datasets/sysu.py WSL_ReID/datasets/llcm.py  # cwd=/kaggle/working/WSL-VIReID
$ git status --short  # cwd=/kaggle/working/WSL-VIReID
M  WSL_ReID/datasets/llcm.py
M  WSL_ReID/datasets/regdb.py
M  WSL_ReID/datasets/sysu.py
M  WSL_ReID/models/__init__.py
A  WSL_ReID/requirements-kaggle.txt
A  WSL_ReID/scripts/apply_kaggle_compat_patches.py
A  WSL_ReID/scripts/check_kaggle_env.py
A  WSL_ReID/scripts/prepare_regdb_kaggle.py
A  WSL_ReID/scripts/run_regdb_dev_baseline.sh
A  WSL_ReID/scrip

remote: This repository moved. Please use the new location:        
remote:   https://github.com/TranTruongMMCII/UIT.CS2309.git        


Branch 'feature/upr-cre' set up to track remote branch 'feature/upr-cre' from 'origin'.
$ git remote -v  # cwd=/kaggle/working/WSL-VIReID
origin	https://github.com/TranTruongMMCII/WSL-VIReID.git (fetch)
origin	https://github.com/TranTruongMMCII/WSL-VIReID.git (push)


remote: 
remote: GitHub found 12 vulnerabilities on TranTruongMMCII/UIT.CS2309's default branch (1 critical, 2 high, 3 moderate, 6 low). To find out more, visit:        
remote:      https://github.com/TranTruongMMCII/UIT.CS2309/security/dependabot        
remote: 
To https://github.com/TranTruongMMCII/WSL-VIReID.git
   f4e3854..e602f86  feature/upr-cre -> feature/upr-cre


CompletedProcess(args=['git', 'remote', '-v'], returncode=0)

## Step 2a completion checklist

Step 2a is done when:

```text
[ ] private fork cloned/pulled using Kaggle Secret GITHUB_TOKEN
[ ] branch feature/upr-cre is active
[ ] Step 0/1 docs are present
[ ] Step 2a scripts are created
[ ] RegDB layout check passes
[ ] optional smoke test passes, or intentionally skipped
[ ] Step 2a commit is pushed to feature/upr-cre
```

After this, the next step is **Step 3: relation diagnostics**. Do not implement UPR-CRE yet.
